# Explore LFP + spike events with SpikeInterface

This notebook loads the Blackrock/Ripple recording in this repo and explores it:

* **`*.ns2`** — analog data @ **1 kHz** (channels labelled `lfp N`) → a SpikeInterface **Recording** (LFP).
* **`*.nev`** — spike events / waveform snippets + digital markers → a SpikeInterface **Sorting**.

> **Note:** 1 kHz LFP cannot be *spike-sorted* — sorting needs the raw broadband stream (~30 kHz, e.g. a `.ns5`/`.ns6` file), which is not in this repo. Here we explore the LFP and the spike units that are already in the `.nev`.

Make sure you launched Jupyter from the project venv: `uv run jupyter lab` (see `README.md`).

In [ ]:
import sys
from pathlib import Path

# Make the helper module in ../scripts importable.
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import blackrock_io as bio

import spikeinterface.full as si
import spikeinterface.widgets as sw

# Interactive matplotlib in the notebook (falls back to inline if ipympl is missing).
try:
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    get_ipython().run_line_magic("matplotlib", "inline")

print("spikeinterface", si.__version__)
print("data set:", bio.find_blackrock_base().name)

## 1. What streams are in the file set?

In [ ]:
bio.list_streams()

## 2. LFP recording (`.ns2`)

In [ ]:
recording = bio.read_lfp()
recording

In [ ]:
# Plot a short window of the raw LFP.
sw.plot_traces(recording, time_range=[0, 5], backend="matplotlib")

In [ ]:
# Optional: band-pass into a typical LFP band and compare.
import spikeinterface.preprocessing as spre

lfp_filtered = spre.bandpass_filter(recording, freq_min=1, freq_max=300)
sw.plot_traces(lfp_filtered, time_range=[0, 5], backend="matplotlib")

## 3. Spike units (`.nev`)

Blackrock unit-id convention: `0` = unsorted threshold crossings, `1..n` = online-sorted units, `255` = noise.

In [ ]:
sorting = bio.read_spikes()
print("units:", list(sorting.get_unit_ids()))
sorting

In [ ]:
# Spike raster across all units.
sw.plot_rasters(sorting)

In [ ]:
# Mean firing rate per unit.
fs = sorting.get_sampling_frequency()
for unit in sorting.get_unit_ids():
    train = sorting.get_unit_spike_train(unit)
    dur = train[-1] / fs if len(train) else 0
    rate = len(train) / dur if dur else 0
    print(f"unit {unit!s:>4}: {len(train):6d} spikes, {rate:6.2f} Hz")

## 4. Digital / serial event markers (`.nev`)

Trellis/Blackrock files often store stimulus or trial markers as digital events. These are read straight from neo.

In [ ]:
events = bio.read_events()
for ev in events:
    print(f"{ev['name']!r}: {len(ev['times'])} events")
if not events:
    print("(no event channels in this file)")

## Where to go next

* **LFP analysis** — power spectra, time-frequency, band-power; use `recording.get_traces(...)` to pull arrays.
* **Peri-event analysis** — align spikes/LFP to the event times from section 4 (PSTHs, evoked LFP).
* **Quality metrics on existing units** — build a `SortingAnalyzer` to compute correlograms, ISI violations, firing-rate stats:
  ```python
  analyzer = si.create_sorting_analyzer(sorting, recording)
  analyzer.compute(["correlograms", "isi_histograms"])
  ```
* **Spike sorting** — only possible if you add the **raw broadband** stream (`.ns5`/`.ns6`, ~30 kHz). See `README.md`.